# REDDIT-MULTI-5K MDL diagnostics

This notebook runs the production `buhito.mdl` implementation on an external
TU-format dataset. It keeps two analyses separate:

- **honest MDL selection**, where the empty dictionary may win;
- **forced diagnostic selection**, which constructs reversible contractions
  even when the bit savings are negative.

The dataset is not stored in Git.

In [ ]:
from pathlib import Path
import math
import os

from buhito.datasets import load_tu_dataset
from buhito.mdl import MDLGraphCompressor, labeled_isomorphic

In [ ]:
DATA_ROOT = Path(os.environ.get("BUHITO_TU_ROOT", "data/TUDataset"))
DATASET = "REDDIT-MULTI-5K"
FIT_SIZE = int(os.environ.get("BUHITO_REDDIT_FIT_SIZE", "100"))
EVAL_SIZE = int(os.environ.get("BUHITO_REDDIT_EVAL_SIZE", "50"))

print("Data root:", DATA_ROOT.resolve())
print("Fit/eval sizes:", FIT_SIZE, EVAL_SIZE)

In [ ]:
dataset = load_tu_dataset(
    DATA_ROOT,
    DATASET,
    node_label_mode="none",
    edge_label_mode="none",
)

fit_graphs = dataset.graphs[:FIT_SIZE]
eval_graphs = dataset.graphs[FIT_SIZE:FIT_SIZE + EVAL_SIZE]
print(f"Loaded {len(dataset.graphs)} graphs; using {len(fit_graphs)} fit and {len(eval_graphs)} eval graphs.")

## Honest selection

In [ ]:
honest = MDLGraphCompressor(
    graphlet_sizes=(3,),
    n_rules=2,
    min_graph_support=5,
    min_occurrences=20,
    max_candidates=10,
    node_label_keys=None,
    edge_label_keys=None,
    selector="sparse",
    dictionary_selection="best",
    cache_dir="artifacts/notebook_cache/reddit_honest",
    validate=True,
    progress=True,
)
honest.fit(fit_graphs)
honest_result = honest.transform(eval_graphs)

honest_result.report

In [ ]:
display(honest.candidate_table_)
display(honest.dictionary_path_)
display(honest_result.per_graph.head())

## Forced diagnostic dictionary

In [ ]:
diagnostic = MDLGraphCompressor(
    graphlet_sizes=(3,),
    n_rules=2,
    min_graph_support=5,
    min_occurrences=20,
    max_candidates=10,
    node_label_keys=None,
    edge_label_keys=None,
    selector="all_eligible",
    dictionary_selection="fixed",
    min_rule_savings_bits=-math.inf,
    cache_dir="artifacts/notebook_cache/reddit_forced",
    validate=True,
    progress=True,
)
diagnostic.fit(fit_graphs)
diagnostic_result = diagnostic.transform(eval_graphs)

diagnostic_result.report

In [ ]:
display(diagnostic.dictionary_frame())
display(
    diagnostic_result.per_graph[
        [
            "graph_index",
            "selected_occurrences",
            "node_reduction_fraction",
            "edge_reduction_fraction",
            "boundary_bits",
            "gross_gain_bits",
        ]
    ].head(20)
)

In [ ]:
decoded = diagnostic_result.decoded_graphs()
assert all(
    labeled_isomorphic(original, reconstructed)
    for original, reconstructed in zip(eval_graphs, decoded)
)
print(f"Exact topology decoding verified for {len(decoded)} REDDIT graphs.")

## Interpretation

The forced run is diagnostic, not evidence of positive compression. It reveals
how much node/edge reduction is available and whether boundary-port metadata is
the dominant cost. Report its `net_savings_bits` together with the reductions.